# Deliverable 4 — Train and compare segmentation models

Trains the project's Small U-Net and U-Net on paired SAR imagery, evaluates each epoch on the validation split, and retains the best-Dice checkpoints. Checkpoints and training history are written to this submission folder, not the project. Set `DATA_ROOT` if needed. Training uses the project's configured epoch and batch-size defaults.

In [ ]:
from pathlib import Path
import sys
import json
import torch
from torch.utils.data import DataLoader

def locate(name):
    for candidate in (Path.cwd(), Path.cwd().parent / name, Path.cwd().parent.parent / name):
        if candidate.name == name and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate sibling folder: {name}")

SUBMISSION_DIR = locate("oil_spill_detection_submission")
candidates = (Path.cwd(), Path.cwd().parent / "oil_spill_detection", Path.cwd().parent.parent / "oil_spill_detection")
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src" / "train.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project folder. Open the notebook from the project or submission folder.")
sys.path.insert(0, str(PROJECT_ROOT))
from config import BATCH_SIZE, EPOCHS, IMAGE_SIZE, SEED
from src.data import SARSegmentationDataset
from src.models import SmallUNet, UNet
from src.train import seed_everything, train_one

DATA_ROOT = PROJECT_ROOT / "data" / "raw"  # Change this if your dataset is elsewhere.
MODEL_OUTPUT_DIR = SUBMISSION_DIR / "deliverable_4_outputs" / "models"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Training data:", DATA_ROOT)
print("Checkpoints and histories:", MODEL_OUTPUT_DIR)

In [ ]:
seed_everything(SEED)
train_dataset = SARSegmentationDataset(DATA_ROOT, "train", IMAGE_SIZE, augment=True)
validation_dataset = SARSegmentationDataset(DATA_ROOT, "val", IMAGE_SIZE, augment=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Training pairs: {len(train_dataset)}; validation pairs: {len(validation_dataset)}")
print(f"Epochs: {EPOCHS}; batch size: {BATCH_SIZE}; image size: {IMAGE_SIZE}")

In [ ]:
results = {}
for name, model in (("SmallUNet", SmallUNet()), ("UNet", UNet())):
    print(f"\nTraining {name}")
    results[name] = train_one(
        name=name,
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        epochs=EPOCHS,
        device=device,
        outdir=MODEL_OUTPUT_DIR,
    )

comparison_path = SUBMISSION_DIR / "deliverable_4_outputs" / "validation_model_comparison.json"
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
display(results)
print("Validation comparison saved:", comparison_path)
print("Best model by validation Dice:", max(results, key=lambda model_name: results[model_name]["dice"]))